=====================================
# BCRA-DATA
=====================================

In [1]:
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import requests
import urllib3
import io
import re
import numpy as np
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import calendar 
import warnings
import json
import eikon as ek

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

ModuleNotFoundError: No module named 'eikon'

=====================================
### ITCRM
=====================================

In [ ]:
# Desactivar advertencias SSL del BCRA
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def obtener_itcrm_actualizado():
    print("1. Entrando a la web del ITCRM del BCRA...")
    
    # URL de la página donde están los datos
    url_web = "https://www.bcra.gob.ar/PublicacionesEstadisticas/Indices_tipo_cambio_multilateral.asp"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }
    
    try:
        # Paso 1: Buscar la URL del Excel en la página
        res_web = requests.get(url_web, headers=headers, verify=False, timeout=15)
        res_web.raise_for_status()
        
        # Buscamos cualquier link que sea ITCRMSerie.xls o .xlsx
        links = re.findall(r'href="([^"]+ITCRMSerie\.xlsx?)"', res_web.text, re.IGNORECASE)
        
        if links:
            ruta_parcial = links[0]
            url_excel = "https://www.bcra.gob.ar" + ruta_parcial if ruta_parcial.startswith("/") else ruta_parcial
            print(f"✅ ¡Excel del ITCRM encontrado!: {url_excel}")
        else:
            # Fallback de seguridad
            url_excel = "https://www.bcra.gob.ar/archivos/Pdfs/PublicacionesEstadisticas/ITCRMSerie.xlsx"
            print(f"⚠️ Link no encontrado en HTML, usando enlace directo: {url_excel}")

        # Paso 2: Descargar el Excel
        print("2. Descargando el archivo Excel a la memoria...")
        res_excel = requests.get(url_excel, headers=headers, verify=False, timeout=15)
        res_excel.raise_for_status()
        
        # Paso 3: Leer los datos con Pandas
        print("3. Procesando los datos del Tipo de Cambio Real Multilateral...")
        
        # --- LA CORRECCIÓN ESTÁ AQUÍ ---
        # Dejamos que Pandas decida el motor automáticamente (openpyxl para xlsx)
        try:
            df_raw = pd.read_excel(io.BytesIO(res_excel.content), header=None)
        except ValueError:
            # Si llega a quejarse porque en realidad era un .xls viejo, usamos el motor viejo
            df_raw = pd.read_excel(io.BytesIO(res_excel.content), engine='xlrd', header=None)
        # -------------------------------

        # El archivo del ITCRM tiene texto inútil arriba. Buscamos dónde empieza la tabla.
        columna_textos = df_raw[0].astype(str).str.lower()
        idx_inicio = columna_textos[columna_textos.str.contains("período|fecha|itcrm")].index
        
        if len(idx_inicio) > 0:
            fila_nombres = idx_inicio[0]
            # Cortamos el dataframe desde donde empiezan los datos reales
            df_itcrm = df_raw.iloc[fila_nombres + 1:].copy()
        else:
            df_itcrm = df_raw.iloc[2:].copy()
            
        # Nos quedamos SOLO con las dos primeras columnas (Fecha y Valor ITCRM)
        df_itcrm = df_itcrm.iloc[:, [0, 1]]
        df_itcrm.columns = ['Fecha', 'ITCRM_Valor']
        
        # Limpieza final: Eliminar filas vacías o textos basura al final del Excel
        df_itcrm = df_itcrm.dropna(subset=['Fecha', 'ITCRM_Valor'])
        df_itcrm = df_itcrm[pd.to_numeric(df_itcrm['ITCRM_Valor'], errors='coerce').notnull()]
        
        # Convertir a Fecha de Python y fijarla como índice
        df_itcrm['Fecha'] = pd.to_datetime(df_itcrm['Fecha'], errors='coerce')
        df_itcrm = df_itcrm.dropna(subset=['Fecha']) 
        df_itcrm['ITCRM_Valor'] = df_itcrm['ITCRM_Valor'].astype(float)
        
        df_itcrm = df_itcrm.set_index('Fecha').sort_index()
        
        print(f"✅ ¡Éxito! Se descargó la serie desde {df_itcrm.index.min().strftime('%d/%m/%Y')} hasta {df_itcrm.index.max().strftime('%d/%m/%Y')}")
        return df_itcrm

    except Exception as e:
        print(f"❌ Error al intentar extraer los datos del ITCRM: {e}")
        return pd.DataFrame()

# ==========================================
# EJECUCIÓN
# ==========================================
df_mi_itcrm = obtener_itcrm_actualizado()

display(df_mi_itcrm.tail())
df_mi_itcrm.to_csv("ITCRM/ITCRM.csv")

1. Entrando a la web del ITCRM del BCRA...
✅ ¡Excel del ITCRM encontrado!: https://www.bcra.gob.ar/archivos/Pdfs/PublicacionesEstadisticas/ITCRMSerie.xlsx
2. Descargando el archivo Excel a la memoria...
3. Procesando los datos del Tipo de Cambio Real Multilateral...
✅ ¡Éxito! Se descargó la serie desde 01/01/1997 hasta 07/04/2026


,ITCRM_Valor
Fecha,
2026-04-03,85.532903
2026-04-04,85.475893
2026-04-05,85.418921
2026-04-06,85.224198
2026-04-07,85.218792


=====================================
### REM
=====================================

In [ ]:
# Desactivar advertencias de SSL
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def obtener_rem_completo():
    print("1. Entrando a la página del BCRA como 'Scraper'...")
    url_home = "https://www.bcra.gob.ar/PublicacionesEstadisticas/Relevamiento_Expectativas_de_Mercado.asp"
    
    try:
        res_home = requests.get(url_home, verify=False, timeout=15)
        res_home.raise_for_status()
        
        links = re.findall(r'href="([^"]+\.xlsx)"', res_home.text, re.IGNORECASE)
        url_excel = None
        for link in links:
            if "hist" in link.lower() or "result" in link.lower():
                url_excel = "https://www.bcra.gob.ar" + link if link.startswith("/") else link
                break
                
        if not url_excel:
            print("❌ No se encontró el Excel en la web.")
            return pd.DataFrame()
            
        print(f"✅ ¡URL capturada!: {url_excel}")
        
    except Exception as e:
        print(f"❌ Error en la web: {e}")
        return pd.DataFrame()

    print("2. Descargando el archivo Excel a la memoria...")
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        response = requests.get(url_excel, headers=headers, verify=False, timeout=15)
        response.raise_for_status() 
        excel_data = io.BytesIO(response.content)
        
        print("3. Extrayendo TODA la macroeconomía del REM...")
        
        df_raw = pd.read_excel(excel_data, header=None)
        fechas = df_raw.iloc[1, 1:].values
        nombres_variables = df_raw[0].astype(str).str.strip()
        
        def limpiar_numero_excel(x):
            if pd.isna(x) or str(x).strip() == '' or str(x).strip().lower() == 'nan': return np.nan
            if isinstance(x, (int, float)): return float(x)
            try: return float(str(x).replace('.', '').replace(',', '.'))
            except: return np.nan

        # =========================================================
        # EL MOTOR DE EXTRACCIÓN: Diccionario de variables y regex
        # =========================================================
        diccionario_busqueda = {
            'Proy_Inflacion_12M': r"IPC nivel general.*Próx\. 12 meses",
            'Proy_Inflacion_24M': r"IPC nivel general.*Próx\. 24 meses",
            'Proy_Inflacion_Nucleo_12M': r"IPC núcleo.*Próx\. 12 meses",
            'Proy_Dolar_12M': r"Tipo de cambio nominal.*Próx\. 12 meses",
            'Proy_Tasa_Interes_12M': r"Tasa de interés.*Próx\. 12 meses"
        }
        
        # Diccionario base para armar el nuevo DataFrame
        datos_extraidos = {'Mes_Relevamiento': fechas}
        
        # Iteramos dinámicamente por cada variable que queremos
        for nombre_columna, patron_regex in diccionario_busqueda.items():
            match = nombres_variables[nombres_variables.str.contains(patron_regex, regex=True, na=False)]
            if not match.empty:
                idx = match.index[0]
                fila_mediana = df_raw.iloc[idx + 1, 1:].values # +1 es la fila de la "Mediana"
                datos_extraidos[nombre_columna] = [limpiar_numero_excel(x) for x in fila_mediana]
            else:
                print(f"⚠️ Advertencia: No se encontró la fila para {nombre_columna}")
                datos_extraidos[nombre_columna] = [np.nan] * len(fechas)

        # Armamos el DataFrame
        df_rem = pd.DataFrame(datos_extraidos)
        df_rem = df_rem.dropna(subset=['Mes_Relevamiento'])
        
        # Convertidor de Fechas robusto
        def parsear_fecha(f):
            if pd.isna(f) or str(f).lower() == 'nan': return pd.NaT
            if isinstance(f, pd.Timestamp) or hasattr(f, 'to_pydatetime'): return pd.to_datetime(f)
            f_str = str(f).strip().lower()
            if f_str.startswith('20') and len(f_str) >= 10: return pd.to_datetime(f_str[:10], errors='coerce')
            meses_es_to_en = {'ene': 'Jan', 'feb': 'Feb', 'mar': 'Mar', 'abr': 'Apr', 'may': 'May', 'jun': 'Jun', 'jul': 'Jul', 'ago': 'Aug', 'sep': 'Sep', 'oct': 'Oct', 'nov': 'Nov', 'dic': 'Dec'}
            mes = f_str[:3]
            anio = f_str[-2:]
            mes_en = meses_es_to_en.get(mes)
            if mes_en: return pd.to_datetime(f"{mes_en}-{anio}", format="%b-%y", errors='coerce')
            return pd.to_datetime(f_str, errors='coerce')

        df_rem['Fecha'] = df_rem['Mes_Relevamiento'].apply(parsear_fecha)
        df_rem = df_rem.dropna(subset=['Fecha']).drop(columns=['Mes_Relevamiento']).set_index('Fecha')
        
        # =========================================================
        # CÁLCULOS FINANCIEROS DE VALOR AGREGADO
        # =========================================================
        # Tasa Real (Ecuación de Fisher: ((1 + i) / (1 + pi)) - 1)
        if 'Proy_Tasa_Interes_12M' in df_rem.columns and 'Proy_Inflacion_12M' in df_rem.columns:
            tasa = df_rem['Proy_Tasa_Interes_12M'] / 100
            infla = df_rem['Proy_Inflacion_12M'] / 100
            df_rem['Proy_Tasa_Real_12M'] = ((1 + tasa) / (1 + infla) - 1) * 100
        
        print("✅ ¡Éxito total! Macroeconomía completa procesada.")
        return df_rem

    except Exception as e:
        print(f"❌ Error al extraer los datos del REM: {e}")
        return pd.DataFrame()

# Ejecución y muestra
df_mi_rem = obtener_rem_completo()
df_mi_rem.to_csv("REM/REM.csv", index=False)
display(df_mi_rem.tail())

1. Entrando a la página del BCRA como 'Scraper'...
✅ ¡URL capturada!: https://www.bcra.gob.ar/archivos/Pdfs/PublicacionesEstadisticas/informes/historico-relevamiento-expectativas-mercado.xlsx
2. Descargando el archivo Excel a la memoria...
3. Extrayendo TODA la macroeconomía del REM...
✅ ¡Éxito total! Macroeconomía completa procesada.


,Proy_Inflacion_12M,Proy_Inflacion_24M,Proy_Inflacion_Nucleo_12M,Proy_Dolar_12M,Proy_Tasa_Interes_12M,Proy_Tasa_Real_12M
Fecha,,,,,,
2025-10-31,20.786459,11.7,20.600000,1697.364287,22.805000,1.671165
2025-11-30,21.041731,12.0,20.000000,1667.900000,22.000000,0.791685
2025-12-31,20.149372,12.7,20.520000,1752.500000,21.000000,0.707975
2026-01-31,20.972204,14.6,19.900000,1767.565000,21.774853,0.663498
2026-02-28,22.300000,15.4,21.859547,1747.516726,22.145000,-0.126738
